# Batch Creating

This notebook is the V2 companion of `../../publish/publish_batch.ipynb`. It shows how to create many test results, steps, and measurements with one call.

V2 does not have `publish_condition_batch` or `publish_measurement_batch`. Instead, it has:
- `batch_create_test_results`: creates test results, and any steps, conditions, and measurements nested inside them.
- `batch_create_steps`: creates steps, and any child steps, conditions, and measurements nested inside them.
- `batch_create_measurements`: creates measurements. Each measurement may target a different step.

Each batch call is **atomic**: either every item is created, or nothing is created.

In [ ]:
# Perform example-specific setup. This is not needed when writing production code.
# Set MDS_V2_GRPC_TARGET (for example, "[::1]:59606") to connect to a specific V2 service.
# When it is not set, the clients use discovery to find the service.
import os

import grpc

grpc_target = os.environ.get("MDS_V2_GRPC_TARGET", "")
# Remove gRPC's 4 MB receive limit so that large values, such as long waveforms, can be read.
grpc_options = [("grpc.max_receive_message_length", -1)]
grpc_channel = grpc.insecure_channel(grpc_target, options=grpc_options) if grpc_target else None

from ni.datastore.data.v2 import DataStoreClient

data_store_client = DataStoreClient(grpc_channel=grpc_channel)

## Create a Test Result with Nested Steps and Measurements

A test result can include its steps, and a step can include its conditions, measurements, and child steps. One `batch_create_test_results` call creates the whole tree.

In [ ]:
from ni.datastore.data.v2 import Condition, Measurement, Step, TestResult

test_result = TestResult(
    name="Batch Create Example",
    steps=[
        Step(
            name="Example Parent Step",
            conditions=[Condition("Ambient Temperature", 25.0, condition_type="Environment", units="degC")],
            measurements=[Measurement(name="Example Parent Measurement", value=1.0, units="V")],
            children=[
                Step(
                    name="Example Child Step",
                    measurements=[Measurement(name="Example Child Measurement", value=2.0, units="V")],
                ),
            ],
        ),
    ],
)

[test_result_id] = data_store_client.batch_create_test_results([test_result])
print(f"Created Test Result: {test_result_id}")

created_steps = data_store_client.query_steps(f"$filter=TestResultId eq {test_result_id}")
created_measurements = data_store_client.query_measurements(f"$filter=TestResultId eq {test_result_id}")
for step in created_steps:
    kind = "child" if step.parent_step_id else "parent"
    print(f"Created {kind} step: {step.name}")
for measurement in created_measurements:
    print(f"Created measurement: {measurement.name} = {measurement.value}")

assert len(created_steps) == 2
assert len(created_measurements) == 2

## Batch Creating Steps

`batch_create_steps` creates several steps at once. Each step names the test result that owns it.

In [ ]:
step_ids = data_store_client.batch_create_steps(
    [
        Step(
            name="Example Step 1",
            test_result_id=test_result_id,
            conditions=[Condition("Supply Voltage", 5.0, condition_type="Setup", units="V")],
        ),
        Step(
            name="Example Step 2",
            test_result_id=test_result_id,
            conditions=[Condition("Supply Voltage", 12.0, condition_type="Setup", units="V")],
        ),
    ]
)
print(f"Created Steps: {step_ids}")

## Batch Creating Measurements

`batch_create_measurements` creates several measurements at once. The measurements can have different names, different value types, and different steps.

In [ ]:
from datetime import timezone

import hightime as ht
import numpy as np
from nitypes.scalar import Scalar
from nitypes.vector import Vector
from nitypes.waveform import AnalogWaveform, Timing

waveform = AnalogWaveform(
    sample_count=4,
    raw_data=np.array([0.0, 0.25, 0.5, 0.75], dtype=np.float64),
    timing=Timing.create_with_regular_interval(
        ht.timedelta(seconds=1e-3),
        ht.datetime.now(timezone.utc),
    ),
)

measurement_ids = data_store_client.batch_create_measurements(
    [
        Measurement(name="Example Float Measurement", value=Scalar(4.98, "V"), step_id=step_ids[0]),
        Measurement(name="Example String Measurement", value="nominal", step_id=step_ids[0]),
        Measurement(name="Example Vector Measurement", value=Vector([1.0, 1.25, 1.5], units="V"), step_id=step_ids[1]),
        Measurement(name="Example AnalogWaveform Measurement", value=waveform, step_id=step_ids[1]),
    ]
)
print(f"Created Measurements: {measurement_ids}")

The returned IDs are in the same order as the request. Read the values back with `read_measurement_value`:

In [ ]:
for measurement_id in measurement_ids:
    measurement = data_store_client.get_measurement(measurement_id)
    value = data_store_client.read_measurement_value(measurement)
    print(f"{measurement.name}: {value}")

read_back_waveform = data_store_client.read_measurement_value(
    data_store_client.get_measurement(measurement_ids[3]), expected_type=AnalogWaveform
)
assert read_back_waveform.raw_data.tolist() == [0.0, 0.25, 0.5, 0.75]

## Batches Are Atomic

If any item in a batch fails, nothing in the batch is created. Below, the second measurement names a step that does not exist, so the first measurement is not created either.

In [ ]:
import grpc

try:
    data_store_client.batch_create_measurements(
        [
            Measurement(name="Example Rollback Measurement", value=1.0, step_id=step_ids[0]),
            Measurement(
                name="Example Rollback Measurement",
                value=2.0,
                step_id="00000000-0000-0000-0000-000000000000",
            ),
        ]
    )
except grpc.RpcError as e:
    print(f"Batch failed: {e.details()}")

rolled_back = data_store_client.query_measurements("$filter=Name eq 'Example Rollback Measurement'")
print(f"Measurements created by the failed batch: {len(rolled_back)}")
assert len(rolled_back) == 0

## Clean-Up

Close the `DataStoreClient` and tear down the example-specific setup when done.

In [ ]:
data_store_client.close()

# Perform example-specific cleanup. This is not needed when writing production code.
if grpc_channel is not None:
    grpc_channel.close()